<a href="https://colab.research.google.com/github/Hibashanti/Time-Series-Analysis/blob/main/Chicago_Crimes_Core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chicago Crime Data

- Author : Hiba Shanti

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt


# Load the Data
- Integrating multiple years of Chicago Crime files into one dataset.  

In [ ]:
folder_path = "/content/drive/MyDrive/Axsos Academy/AXSOSACADEMY-1/Time Series Data/Data/Data"
files = os.listdir(folder_path)

print(files)  # to confirm your files

In [ ]:
df_list = []

for file in files:
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        df_temp = pd.read_csv(file_path)
        df_list.append(df_temp)

# Combine all dataframes
df = pd.concat(df_list, ignore_index=True)

print(df.shape)
df.head()

In [ ]:
# Convert the datt column to a datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed')
df=df.set_index('Date')
df=df.drop(columns="ID")
df.head()

## Topic 1) Comparing Police Districts


##### Which district had the most crimes in 2022?


In [ ]:
#1- Calculate the number of crimes for each district yearly
crimes=df.groupby("District").resample("YE").size()
crimes

In [ ]:
# 2- Get the crimes for all discrits in 2022
crimes_22=crimes.loc[(slice (None), "2022")]
crimes_22

In [ ]:
# Defining Districs with most crimes in 2022
most_crimes_22_number=crimes_22.max()
most_crimes_22_name=crimes_22.idxmax()
print(f"max crime number in 2022 is {most_crimes_22_number},in {most_crimes_22_name} discrit")

#Which District had the least number of crimes?


In [ ]:
# Defining Districs with most crimes in 2022

least_crimes_22_number=crimes_22.min()
least_crimes_22_name=crimes_22.idxmin()
print(f"min crime number in 2022 is {least_crimes_22_number},in {least_crimes_22_name} discrit")


In [ ]:
# Visualize the ordered discrits in terms of crime numbers in 2022.
crimes_order=crimes_22.sort_values(ascending=False)
ax=crimes_order.plot(kind="bar")

## Topic 2) Crimes Across the Years:

In [ ]:
#Is the total number of crimes increasing or decreasing across the years
crimes_trend=df.resample("YE").size()

fig,ax=plt.subplots()
sns.lineplot(x=crimes_trend.index.year, y=crimes_trend.values,ax=ax);
ax.set_xlabel("Year",rotation=45)
ax.set_ylabel("Number of Crimes")


#Chicago crimes have generally decreased over time,with a slight fluctuation.
#The sharp drop in crimes after 2023 is due to incomplete or missing data for the most recent year, not an actual decline.


In [ ]:
#Are there any individual crimes that are doing the opposite (e.g., decreasing when overall crime is increasing or vice-versa)?
# 1- define the individual crimes during the years
time_bucket=pd.Grouper(freq="YE")
individual_crimes=df.groupby(["Primary Type", time_bucket]).size()

In [ ]:
# Visualize the individual crimes during the years
unstacked=individual_crimes.unstack(level=0)
individual_trend=unstacked.plot(figsize=(12,6))
individual_trend.set_xlabel("Years")
individual_trend.set_ylabel("Trend")
plt.legend(bbox_to_anchor=(1.05,1),loc="upper left");


# Topic 3) Comparing AM vs. PM Rush Hour:


- Are crimes more common during AM rush hour or PM rush hour?
  - You can consider any crime that occurred between 7 AM - 10 AM as AM rush hour
  - You can consider any crime that occurred between 4 - 7 PM as PM rush hour.

In [ ]:
# 1-Define new column for the Hours in the dataset
df["Hour"]=df.index.hour
df["Hour"]

In [ ]:
# 2-Identify each rushing hour group
am_rush=df[(df["Hour"] >= 7) & (df["Hour"]<= 10)]
pm_rush=df[(df["Hour"] >= 16) & (df["Hour"]<=19)]

In [ ]:
# 3-Address the length of each group
am_count=len(am_rush)
pm_count=len(pm_rush)

print(f'The Number of crimes in AM rush hour = {am_count}, and in PM rush hour = {pm_count}')

- Crimes are more common to occure during PM rush hour with a total number of 1,641,051 compared with a total of 1,097,647 in AM rush hour.


In [ ]:
#What are the top 5 most common crimes during AM rush hour? What are the top 5 most common crimes during PM rush hour?
top_5_am=am_rush["Primary Type"].value_counts()
top_5_am.head(5)

In [ ]:
#What are the top 5 most common crimes during PM rush hour?
top_5_pm=pm_rush["Primary Type"].value_counts()
top_5_pm.head(5)

In [ ]:
#Are Motor Vehicle Thefts more common during AM rush hour or PM Rush Hour?
# specify the Motor Vehicle Thefts
mvt_am=am_rush[am_rush["Primary Type"]== "MOTOR VEHICLE THEFT"]
mvt_pm=pm_rush[pm_rush["Primary Type"]== "MOTOR VEHICLE THEFT"]

In [ ]:
# Define the length of each period
mvt_am_count=len(mvt_am)
mvt_pm_count=len(mvt_pm)
print(f' number of Motor Vehicle Thefts in AM rush hour= {mvt_am_count},number of Motor Vehicle Thefts in PM rush hour= {mvt_pm_count}')

- The number of Motor Vehicle Thefts is more common in the PM rush hour.

 # 4) Comparing Months:


In [ ]:
# Adding a Month column to the dataset
df["Month"]=df.index.month
df["Month"]

In [ ]:
# What months have the most crime? What months have the least?
monthly_count=df["Month"].value_counts()
most_month=monthly_count.idxmax()
least_month=monthly_count.idxmin()
print(f'the months with the most crime is {most_month}, and the month with the least is {least_month} ')

In [ ]:
#Are there any individual crimes that do not follow this pattern? If so, which crimes?
monthly_crimes=df.groupby(["Primary Type", "Month"]).size()

In [ ]:
# Visualize the data for each crime during the months
unstacked=monthly_crimes.unstack(level=0)
unstacked.plot()
plt.legend(bbox_to_anchor=(1.05,1),loc="upper left");

In [ ]:
# to find the peak for each month,
peak_per_month=unstacked.idxmax()
peak_per_month

In [ ]:
# to find the least point for each month,
least_per_month=unstacked.idxmin()
least_per_month

5) Comparing Holidays:

 What are the top 3 holidays with the largest number of crimes?
  For each of the top 3 holidays with the most crime, what are the top 5 most common crimes on that holiday?

In [ ]:
# What are the top 3 holidays with the largest number of crimes?
جيبي العطل اللي بشيكاغو
احسبي عدد الجريمة في كل وحدة
اعمل value count
طلعي اعلى ٣ عطل